Created by Angelo Orletti Del Rey for its masters in psychiatry at Unifesp - SP - Brasil

It was based in our groups previeous works in cotical striatal connectivity in INPD data base of fmri

In [1]:
import numpy as np
import os, sys
import bids
import nibabel as nib
from scipy.stats import pearsonr
from scipy.stats import chi2
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.genmod.families import NegativeBinomial
from nilearn import plotting as nplot
from nilearn import image as nimg
from nilearn.image import resample_to_img
import matplotlib.pyplot as plt
import argparse
import pandas as pd

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from scipy.linalg import LinAlgError

# Suppress convergence warnings
warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="statsmodels")


c:\Users\angel\Documents\masters-uploaded-github\INPD-neuroimage-cape\.venv-CAPE-neuroimag\lib\site-packages\nilearn\__init__.py:67: FutureWarning: Python 3.7 support is deprecated and will be removed in release 0.12 of Nilearn. Consider switching to Python 3.9 or 3.10.
  _python_deprecation_warnings()


In [2]:
def parse():

        options = argparse.ArgumentParser(description="Run 1st level analysis. Created by ...")
        options.add_argument('-p', '--participants', nargs='+',dest="participants", action='store', type=str, required=False,
                            help='id of subject or list of subjects')
        options.set_defaults(participants=None)
        options.add_argument('-w', '--workdir',dest="workdir", action='store', type=str, required=False,
                            help='the work directory for the project')
        options.set_defaults(workdir=os.environ["ROOTDIR"])
        options.add_argument('-b', '--bidsdir',dest="rawdata", action='store', type=str, required=False,
                            help='the work directory for the project')
        options.set_defaults(rawdata=os.path.join(os.environ["ROOTDIR"],"BIDS"))
        options.add_argument('-d', '--derivatives',dest="derivatives", action='store', type=str, required=True,
                            help='path to fMRIprep directory')
        options.add_argument('-o', '--outputs',dest="output", action='store', type=str, required=True,
                            help='path to fMRIprep directory')
        #print(options.parse_args())

        return options.parse_args()

os.environ["ROOTDIR"] = r'D://'   # seth path
rootdir = os.environ["ROOTDIR"]
if hasattr(sys, "ps1"):
    options = {}
    workdir = os.environ["ROOTDIR"]
    firstleveldir  = os.path.join(workdir,"BIDS","derivatives","first_level_results")
    demographic = os.path.join(workdir,"metadata")
    confounds = os.path.join(workdir,"metadata")
    output = os.path.join(workdir,'second_level_results')
    participants = []

else :
    options = parse()
    participants = options.participants
    workdir = options.workdir
    rawdata = options.rawdata
    derivat = options.derivatives
    output  = options.outputs

print('firstlevel: ', firstleveldir)
    
bidslayout = bids.BIDSLayout(firstleveldir, validate = False) #With validate = True it doesn't find any subjects

if not participants:
    participants = bidslayout.get_subjects() #take all subjects
    #participants_sub = ["sub-" + item for item in participants]

seednames = ['DCPutamen',
            'DorsalCaudate',
            'DRPutamen',
            'InfVentralCaudate',
            'SupVentralCaudate',
            'VRPutamen'
            ]

tmap_allsubj = [] #initialize array for tmaps

print("Loading tmaps for all participants...")
for p in participants:
    #print(f"Subject: {p}")
    p = p.replace("sub-", "")

    # for ses in bidslayout.get_sessions(subject=p):
    #     #print(f"Session: {ses}")                
    
    ses = '1'

    for r in bidslayout.get_runs(subject=p, session=ses):
        #print(f"Run: {r}")

        tmap_metadata = {
            'task': 'rest',
            'suffix': 'tstat',
            'extension': '.txt',
            'run': str(r),
            'session': str(ses),
            'subject': 'sub-'+p,
            'space': 'MNI152NLin2009cAsym',
            'seed': 'SeedtoROI'
        }

        # Save each tmap as a nifti file
        filepath = os.path.join(firstleveldir,tmap_metadata["subject"],"ses-"+tmap_metadata['session'],"func")
        filename = tmap_metadata["subject"] + "_" + \
                    "ses-"+tmap_metadata['session'] + "_" + \
                    "task-"+tmap_metadata['task'] + "_" + \
                    "run-"+tmap_metadata['run'] + "_" + \
                    "space-"+tmap_metadata['space'] + "_" + \
                    "seed-"+tmap_metadata['seed'] + "_" + \
                    tmap_metadata["suffix"] + \
                    tmap_metadata["extension"]
        tmap_path = os.path.join(filepath, filename)

        #load the tmaps for each subject and then append to the tmap_allsubj array
        tmap = pd.read_csv(tmap_path, delimiter=' ', header=None, skiprows=1) #also skips the first row
        tmap.columns = seednames
        tmap_allsubj.append(tmap)

#load socioeconomic and psichometric data for all subjects
print("Loading socioeconomic and psichometric data for all participants...")
socio_psi=pd.read_csv(os.path.join(demographic,'variaveis_analise_conf_nova-exclusão.tsv'), sep='\t')
#substitute in age colunm , to . for decimals if necessary and convert to float
# socio_psi['age'] = socio_psi['age'].str.replace(',', '.').astype(float)

# organinzing the tmaps. Joining the columns of the tmaps for all subjects. So for each subject
# we have to join the first collunm of the first tmap with the first column of the second tmap and so on
# and then join the second column of the first tmap with the second column of the second tmap and so on
# and so on
print("Organizing tmaps...")
tmap_allsubj = pd.concat(tmap_allsubj, axis=1)
tmap_allsubj_organized = {}

for seed in seednames:
    #drop columns with names different from seednames[i]
    tmap_allsubj_organized[seed] = tmap_allsubj.drop(columns=[col for col in tmap_allsubj.columns if col != seed])

# What do we have now: tmap_allsubj_organized is a dictionary with keys being the seednames and values being
# dataframes with the tmaps for all subjects for that seed. This have dimensions of n_ROIs x n_subjects
# Now we have to transpose it to get the data in the format n_subjects x n_ROIs and then join with the
# socioeconomic and psichometric data
# Also, we have to add the sub_id column to the conectivity data to be able to join with the socioeconomic
# and psichometric data
subs = {'sub_id': []}
for p in participants:
    ses = '1'
    if bidslayout.get_runs(subject=p, session=ses) != []:
        subs['sub_id'].append('sub-'+p)

subs = pd.DataFrame(subs)

# create the collunms names for the data frame of rois
roi_names = [f'roi_{i}' for i in range(101)]

for seed in seednames:
    tmap_allsubj_organized[seed] = tmap_allsubj_organized[seed].transpose()
    tmap_allsubj_organized[seed].columns = roi_names
    tmap_allsubj_organized[seed] = tmap_allsubj_organized[seed].reset_index().join(subs, how='inner')
    tmap_allsubj_organized[seed] = tmap_allsubj_organized[seed].drop(columns=['index'])

# Now we have to join the tmaps with the socioeconomic and psichometric data
print("Joining tmaps with socioeconomic and psichometric data...")

for seed in seednames:
    tmap_allsubj_organized[seed] = pd.merge(tmap_allsubj_organized[seed], socio_psi, on='sub_id', how='inner')
    # tmap_allsubj_organized[seed] = tmap_allsubj_organized[seed].drop(columns=['Unnamed: 0'])


firstlevel:  D://BIDS\derivatives\first_level_results
Loading tmaps for all participants...
Loading socioeconomic and psichometric data for all participants...
Organizing tmaps...
Joining tmaps with socioeconomic and psichometric data...


In [3]:
# now we run the regression analysis for each seed. The dependent variable is the connectivity and the independent
# variables are the CAPE scores. The socioeconomic variables are used as confund variables
print("Running regression analysis (salience and defualt mode network)...")

shaeffer_network = pd.read_csv(os.path.join(demographic, 'tpl-MNI152NLin2009cAsym_atlas-Schaefer2018_desc-100Parcels7Networks_dseg.tsv'), delimiter='\t')

#extract the index with the name with SalVentAttn - salience network
salience_index = shaeffer_network[shaeffer_network['name'].str.contains('SalVentAttn')]
#extract the index with the name with Default - default mode network
default_index = shaeffer_network[shaeffer_network['name'].str.contains('Default')]

# running the regression analysis for each roi in the salience network
results_salience, results_default = {}, {}
for cape in ['cape_tot', 'cape_PA_score', 'cape_PI_score', 'cape_BE_score']:
    results_salience[cape], results_default[cape] = {}, {}
    for seed in seednames:
        results_salience[cape][seed], results_default[cape][seed] = {}, {}

        #salience analisys
        for roi in salience_index['index']:

            #building my strign for the fomrula
            formula_salience = f'{cape} ~ roi_{roi} + age + C(gender) + abepscore + C(colection_site) + fd_mean_value'
            
            #running the regression analysis
            model_salience = smf.glm(formula_salience, data=tmap_allsubj_organized[seed],
                                    family=NegativeBinomial())
            # model_salience = smf.ols(formula_salience, data=tmap_allsubj_organized[seed])
            fit_salience = model_salience.fit()

            # Extract desired values
            results_salience[cape][seed][roi] = {
                'coef': fit_salience.params[f'roi_{roi}'],
                'intercept': fit_salience.params['Intercept'],
                't_value': fit_salience.tvalues[f'roi_{roi}'],
                'p_value': fit_salience.pvalues[f'roi_{roi}'],
                # 'R_squared': fit_salience.rsquared,
                # 'Adj_R_squared': fit_salience.rsquared_adj,
                'AIC': fit_salience.aic,
                'BIC': fit_salience.bic
            }
            
        #DMN analisys
        for roi in default_index['index']:

            #building my strign for the fomrula
            formula_default = f'{cape} ~ roi_{roi} + age + C(gender) + abepscore + C(colection_site) + fd_mean_value'
            
            # Run the regression analysis
            model_default = smf.glm(formula_default, data=tmap_allsubj_organized[seed],
                                    family=NegativeBinomial())
            # model_default = smf.ols(formula_default, data=tmap_allsubj_organized[seed])
            fit_default = model_default.fit()

            # Extract desired values
            results_default[cape][seed][roi] = {
                'coef': fit_default.params[f'roi_{roi}'],
                'intercept': fit_default.params['Intercept'],
                't_value': fit_default.tvalues[f'roi_{roi}'],
                'p_value': fit_default.pvalues[f'roi_{roi}'],
                # 'R_squared': fit_default.rsquared,
                # 'Adj_R_squared': fit_default.rsquared_adj,
                'AIC': fit_default.aic,
                'BIC': fit_default.bic
            }

Running regression analysis (salience and defualt mode network)...


c:\Users\angel\Documents\masters-uploaded-github\INPD-neuroimage-cape\.venv-CAPE-neuroimag\lib\site-packages\statsmodels\genmod\generalized_linear_model.py:1809: FutureWarning: The bic value is computed using the deviance formula. After 0.13 this will change to the log-likelihood based formula. This change has no impact on the relative rank of models compared using BIC. You can directly access the log-likelihood version using the `bic_llf` attribute. You can suppress this message by calling statsmodels.genmod.generalized_linear_model.SET_USE_BIC_LLF with True to get the LLF-based version now or False to retainthe deviance version.
  FutureWarning
c:\Users\angel\Documents\masters-uploaded-github\INPD-neuroimage-cape\.venv-CAPE-neuroimag\lib\site-packages\statsmodels\genmod\generalized_linear_model.py:1809: FutureWarning: The bic value is computed using the deviance formula. After 0.13 this will change to the log-likelihood based formula. This change has no impact on the relative rank of

In [4]:
#select a line to see the pvalues of the last fit

# fit_salience.summary()
# results_default['DorsalCaudate']

In [5]:
#correcting the p-values
from statsmodels.stats.multitest import multipletests
for cape in ['cape_tot', 'cape_PA_score', 'cape_PI_score', 'cape_BE_score']:
    for seed in seednames:
        print(f'Salience - {seed}')
        p_values = []

        for roi in results_salience[cape][seed].keys():
            # print(roi)
            # Extract the p-values for the current seed and roi
            p_values.append(results_salience[cape][seed][roi]['p_value'])

        #correct the p-values using the Benjamini-Hochberg method
        corrected_p_values = multipletests(p_values, method='fdr_bh')[1]

        # Update the results with the corrected p-values
        for index, roi in enumerate(results_salience[cape][seed].keys()):
            results_salience[cape][seed][roi]['corrected_p_value'] = corrected_p_values[index]

            if results_salience[cape][seed][roi]['corrected_p_value'] <= 0.05:
                print(f"{cape} roi-{roi} -> corr_p_val {results_salience[cape][seed][roi]['corrected_p_value']}")

        print(f'DMN - {seed}')
        p_values = []

        for roi in results_default[cape][seed].keys():
            # print(roi)
            # Extract the p-values for the current seed and roi
            p_values.append(results_default[cape][seed][roi]['p_value'])

        #correct the p-values using the Benjamini-Hochberg method
        corrected_p_values = multipletests(p_values, method='fdr_bh')[1]

        # Update the results with the corrected p-values
        for index, roi in enumerate(results_default[cape][seed].keys()):
            # print(f'index {index} roi {roi}')
            results_default[cape][seed][roi]['corrected_p_value'] = corrected_p_values[index]

            if results_default[cape][seed][roi]['corrected_p_value'] <= 0.05:
                print(f"{cape} roi-{roi} -> corr_p_val {results_default[cape][seed][roi]['corrected_p_value']}")

Salience - DCPutamen
DMN - DCPutamen
Salience - DorsalCaudate
DMN - DorsalCaudate
Salience - DRPutamen
DMN - DRPutamen
Salience - InfVentralCaudate
DMN - InfVentralCaudate
Salience - SupVentralCaudate
DMN - SupVentralCaudate
Salience - VRPutamen
DMN - VRPutamen
Salience - DCPutamen
DMN - DCPutamen
Salience - DorsalCaudate
DMN - DorsalCaudate
Salience - DRPutamen
DMN - DRPutamen
cape_PA_score roi-42 -> corr_p_val 0.04987817538801672
cape_PA_score roi-44 -> corr_p_val 0.012377880806711997
cape_PA_score roi-46 -> corr_p_val 0.038946886085201894
cape_PA_score roi-49 -> corr_p_val 0.038946886085201894
cape_PA_score roi-93 -> corr_p_val 0.04987817538801672
cape_PA_score roi-94 -> corr_p_val 0.026689634538585114
cape_PA_score roi-98 -> corr_p_val 0.012377880806711997
cape_PA_score roi-99 -> corr_p_val 0.038946886085201894
cape_PA_score roi-100 -> corr_p_val 0.0490841048908856
Salience - InfVentralCaudate
DMN - InfVentralCaudate
Salience - SupVentralCaudate
DMN - SupVentralCaudate
Salience - V

In [6]:
# Merge all regression parameters into a single dataframe grouped by CAPE and seed
all_results = []

for cape in ['cape_tot', 'cape_PA_score', 'cape_PI_score', 'cape_BE_score']:
    # Process salience network results
    for seed in seednames:
        for roi in results_salience[cape][seed].keys():
            # Get the result values as a dictionary
            result_dict = results_salience[cape][seed][roi].to_dict('records')[0] if isinstance(results_salience[cape][seed][roi], pd.DataFrame) else results_salience[cape][seed][roi]
            # Add metadata columns
            result_dict['CAPE'] = cape
            result_dict['seed'] = seed
            result_dict['network'] = 'Salience'
            result_dict['ROI'] = roi
            all_results.append(result_dict)
    
    # Process default mode network results
    for seed in seednames:
        for roi in results_default[cape][seed].keys():
            # Get the result values as a dictionary
            result_dict = results_default[cape][seed][roi].to_dict('records')[0] if isinstance(results_default[cape][seed][roi], pd.DataFrame) else results_default[cape][seed][roi]
            # Add metadata columns
            result_dict['CAPE'] = cape
            result_dict['seed'] = seed
            result_dict['network'] = 'Default'
            result_dict['ROI'] = roi
            all_results.append(result_dict)

# Create dataframe from list of dictionaries
merged_df = pd.DataFrame(all_results)

# Reorder columns for better readability
column_order = ['CAPE', 'seed', 'network', 'ROI', 'coef', 'intercept', 't_value', 'p_value', 'corrected_p_value', 'AIC', 'BIC']
merged_df = merged_df[column_order]

# Save to CSV
merged_df.to_csv(os.path.join(output, 'results_2ndlvl.csv'), index=False)

print(f"Merged dataframe saved to: {os.path.join(output, 'results_2ndlvl.csv')}")
print(f"Shape: {merged_df.shape}")
print(f"\nFirst few rows:")
print(merged_df.head(10))

Merged dataframe saved to: D://second_level_results\results_2ndlvl.csv
Shape: (864, 11)

First few rows:
       CAPE       seed   network  ROI      coef  intercept   t_value  \
0  cape_tot  DCPutamen  Salience   24 -0.021214   1.500351 -0.866520   
1  cape_tot  DCPutamen  Salience   25  0.015389   1.440779  0.543614   
2  cape_tot  DCPutamen  Salience   26 -0.006163   1.487006 -0.216809   
3  cape_tot  DCPutamen  Salience   27  0.003954   1.453229  0.145698   
4  cape_tot  DCPutamen  Salience   28 -0.000695   1.464882 -0.027004   
5  cape_tot  DCPutamen  Salience   29  0.015828   1.447701  0.620878   
6  cape_tot  DCPutamen  Salience   30  0.045488   1.387367  1.599403   
7  cape_tot  DCPutamen  Salience   74  0.014324   1.431615  0.601322   
8  cape_tot  DCPutamen  Salience   75  0.011053   1.424244  0.419645   
9  cape_tot  DCPutamen  Salience   76  0.006836   1.443724  0.297810   

    p_value  corrected_p_value          AIC          BIC  
0  0.386205           0.964538  2166.901592

In [7]:
merged_df

,CAPE,seed,network,ROI,coef,intercept,t_value,p_value,corrected_p_value,AIC,BIC
0,cape_tot,DCPutamen,Salience,24,-0.021214,1.500351,-0.866520,0.386205,0.964538,2166.901592,-1926.359138
1,cape_tot,DCPutamen,Salience,25,0.015389,1.440779,0.543614,0.586707,0.964538,2167.336639,-1925.924091
2,cape_tot,DCPutamen,Salience,26,-0.006163,1.487006,-0.216809,0.828357,0.964538,2167.570689,-1925.690041
3,cape_tot,DCPutamen,Salience,27,0.003954,1.453229,0.145698,0.884160,0.964538,2167.600803,-1925.659927
4,cape_tot,DCPutamen,Salience,28,-0.000695,1.464882,-0.027004,0.978456,0.978456,2167.618289,-1925.642441
...,...,...,...,...,...,...,...,...,...,...,...
859,cape_BE_score,VRPutamen,Default,96,0.009253,0.122984,0.387886,0.698101,0.761564,993.151340,-2026.643088
860,cape_BE_score,VRPutamen,Default,97,0.020542,0.105102,0.720531,0.471198,0.628264,992.821010,-2026.973418
861,cape_BE_score,VRPutamen,Default,98,-0.043139,0.200636,-1.264368,0.206098,0.456989,991.627988,-2028.166440
862,cape_BE_score,VRPutamen,Default,99,-0.034959,0.248910,-0.956643,0.338748,0.508121,992.403156,-2027.391273
